##### Library Imports

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

##### Loading Dataset (Single Participant)

In [1]:
df = pd.read_csv("functional_p18-s15.csv")

# print(df.head())
print(df.shape)

NameError: name 'pd' is not defined

##### PREPROCESSING

In [ ]:
# Remove unnecessary columns
df = df.drop(columns=["start_time", "end_time"])

# Seperate Features and Labels
X = df.drop(columns=['label'])
y = df['label']

# label encoding
encoder = LabelEncoder()
y = encoder.fit_transform(y)
# print(encoder.classes_)

# Train / Validation / Test Split
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.2, # 20% Testing + Validation
    random_state=42,
    stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5, # 10% Testing, 10% Validation
    random_state=42,
    stratify=y_temp
)

# Feature Standardization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

<!-- model -->

##### NEURAL NETWORK

In [ ]:
model = Sequential[(
    Input(shape=(X_train.shape[1],)),

    # Hidden Layer 1
    Dense(128, activation='relu'),
    Dropout(0.3),

    # Hidden Layer 2
    Dense(64, activation='relu'),
    Dropout(0.3),

    # Output Layer 
    Dense(1, activation='sigmoid')
)]



In [ ]:
# Compile Model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
#View the model
model.summary()

In [ ]:
# Early stopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

##### MODEL TRAINING

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1       
)

##### EVALUATION ON TEST SET

In [ ]:
loss, accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0   
)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

In [ ]:
probabilities = model.predict(X_test)
predictions = (probabilities > 0.5).astype(int)

##### PERFORMANCE METRICS

In [ ]:
print("Accuracy :", accuracy_score(y_test, predictions))
print("Precision:", precision_score(y_test, predictions))
print("Recall   :", recall_score(y_test, predictions))
print("F1 Score :", f1_score(y_test, predictions))

In [ ]:
# Confusion MAtrix
cm = confusion_matrix(y_test, predictions)
print(cm)

In [ ]:
# Classification Report
print(classification_report(
    y_test,
    predictions,
    target_names=encoder.classes_
))

In [ ]:
model.save("neural-network.keras")